In [4]:
!pip install pdf2image


[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
!apt-get install poppler-utils # Install poppler-

'apt-get' is not recognized as an internal or external command,
operable program or batch file.


In [6]:
# from google.colab import drive
# drive.mount('/content/drive')

In [7]:
from huggingface_hub import hf_hub_download
from transformers import AutoImageProcessor, TableTransformerForObjectDetection
import torch
from PIL import Image
from pdf2image import convert_from_path # Import to convert PDF to image

file_path = '/content/drive/MyDrive/Colab Notebooks/AAFC_PDCAAS/1-s2.0-S0308814617312839-Lentils.pdf'
# Convert the PDF pages to a list of PIL Images
images = convert_from_path(file_path)

# Initialize the processor and model
image_processor = AutoImageProcessor.from_pretrained("microsoft/table-transformer-detection")
model = TableTransformerForObjectDetection.from_pretrained("microsoft/table-transformer-detection")

# Loop over all pages in the PDF
for page_num, image in enumerate(images):
    print(f"Processing page {page_num + 1}")

    # Process the image
    inputs = image_processor(images=image, return_tensors="pt")
    outputs = model(**inputs)

    # Convert outputs (bounding boxes and class logits) to Pascal VOC format (xmin, ymin, xmax, ymax)
    target_sizes = torch.tensor([image.size[::-1]])  # Ensure image dimensions are in the correct order
    results = image_processor.post_process_object_detection(outputs, threshold=0.8, target_sizes=target_sizes)[0]

    # Loop through the detected tables and save or process the bounding boxes
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        box = [round(i, 2) for i in box.tolist()]
        print(
            f"Detected {model.config.id2label[label.item()]} with confidence "
            f"{round(score.item(), 3)} at location {box}"
        )

        # Crop the detected table from the image based on bounding box
        xmin, ymin, xmax, ymax = map(int, box)
        cropped_table = image.crop((xmin, ymin, xmax, ymax))

        # Optionally, save the cropped table as an image
        cropped_table.save(f"extracted_table_page_{page_num + 1}.png")

        # You can also apply OCR to extract text from the table image if needed


PDFPageCountError: Unable to get page count.
I/O Error: Couldn't open file '/content/drive/MyDrive/Colab Notebooks/AAFC_PDCAAS/1-s2.0-S0308814617312839-Lentils.pdf': No error.


In [27]:
import pandas as pd

In [28]:
# Data for the second table
data1 = {
    "Sample": [
        "Casein",
        "Red Lentil (Untreated)",
        "Red Lentil (Extruded)",
        "Red Lentil (Cooked)",
        "Red Lentil (Baked)",
        "Green Lentil (Untreated)",
        "Green Lentil (Extruded)",
        "Green Lentil (Cooked)",
        "Green Lentil (Baked)"
    ],
    "%DM": [93.56, 92.12, 95.41, 99.57, 97.49, 91.38, 95.13, 99.47, 97.32],
    "%CF": [0.20, 1.78, 1.08, 1.62, 2.34, 1.13, 1.48, 2.06, 2.21],
    "%CP": [86.48, 25.13, 26.86, 26.62, 25.93, 23.93, 24.65, 25.67, 25.44],
    "ASP": [7.78, 2.69, 3.30, 3.31, 3.25, 2.81, 3.05, 3.07, 2.53],
    "THR": [3.35, 0.78, 0.96, 0.96, 1.00, 0.83, 0.89, 0.91, 0.79],
    "SER": [5.64, 1.21, 1.47, 1.56, 1.54, 1.31, 1.36, 1.45, 1.24],
    "GLU": [20.05, 3.55, 4.38, 4.41, 4.47, 3.86, 4.07, 4.09, 3.67],
    "PRO": [9.77, 0.60, 0.79, 0.91, 0.96, 0.90, 0.73, 0.80, 0.53],
    "GLY": [1.35, 0.87, 0.97, 0.85, 1.00, 0.87, 0.91, 0.90, 0.88],
    "ALA": [3.16, 0.98, 1.25, 1.23, 1.29, 1.15, 1.20, 1.20, 1.14],
    "CYS": [0.78, 0.22, 0.24, 0.24, 0.20, 0.20, 0.20, 0.20, 0.18],
    "VAL": [5.02, 0.97, 1.18, 1.21, 1.13, 1.02, 1.10, 1.19, 1.13],
    "MET": [1.45, 0.206, 0.22, 0.21, 0.19, 0.19, 0.20, 0.21, 0.18],
    "ILE": [3.84, 0.77, 0.96, 1.03, 1.00, 0.89, 1.02, 2.05, 1.69],
    "LEU": [8.39, 1.67, 1.88, 2.19, 2.05, 1.75, 1.93, 2.05, 1.69],
    "TYR": [4.83, 0.63, 0.68, 0.71, 0.67, 0.67, 0.64, 0.67, 0.57],
    "PHE": [4.59, 1.09, 1.33, 1.43, 1.26, 1.17, 1.24, 1.32, 1.06],
    "HIS": [2.74, 0.60, 0.79, 0.77, 0.80, 0.66, 0.73, 0.71, 0.67],
    "LYS": [6.96, 1.43, 1.81, 1.83, 1.57, 1.61, 2.14, 2.28, 1.37],
    "ARG": [3.12, 1.88, 2.01, 2.28, 2.30, 2.22, 2.11, 2.21, 1.88],
    "TRP": [1.08, 0.20, 0.22, 0.22, 0.20, 0.18, 0.20, 0.21, 0.20]
}

# Creating DataFrame for table 2
df1 = pd.DataFrame(data1)
print("\nTable 1:")
print(df1)



Table 1:
                     Sample    %DM   %CF    %CP   ASP   THR   SER    GLU  \
0                    Casein  93.56  0.20  86.48  7.78  3.35  5.64  20.05   
1    Red Lentil (Untreated)  92.12  1.78  25.13  2.69  0.78  1.21   3.55   
2     Red Lentil (Extruded)  95.41  1.08  26.86  3.30  0.96  1.47   4.38   
3       Red Lentil (Cooked)  99.57  1.62  26.62  3.31  0.96  1.56   4.41   
4        Red Lentil (Baked)  97.49  2.34  25.93  3.25  1.00  1.54   4.47   
5  Green Lentil (Untreated)  91.38  1.13  23.93  2.81  0.83  1.31   3.86   
6   Green Lentil (Extruded)  95.13  1.48  24.65  3.05  0.89  1.36   4.07   
7     Green Lentil (Cooked)  99.47  2.06  25.67  3.07  0.91  1.45   4.09   
8      Green Lentil (Baked)  97.32  2.21  25.44  2.53  0.79  1.24   3.67   

    PRO   GLY  ...   VAL    MET   ILE   LEU   TYR   PHE   HIS   LYS   ARG  \
0  9.77  1.35  ...  5.02  1.450  3.84  8.39  4.83  4.59  2.74  6.96  3.12   
1  0.60  0.87  ...  0.97  0.206  0.77  1.67  0.63  1.09  0.60  1.43  1.88  

In [11]:
df1

,Sample,%DM,%CF,%CP,ASP,THR,SER,GLU,PRO,GLY,...,VAL,MET,ILE,LEU,TYR,PHE,HIS,LYS,ARG,TRP
0,Casein,93.56,0.20,86.48,7.78,3.35,5.64,20.05,9.77,1.35,...,5.02,1.450,3.84,8.39,4.83,4.59,2.74,6.96,3.12,1.08
1,Red Lentil (Untreated),92.12,1.78,25.13,2.69,0.78,1.21,3.55,0.60,0.87,...,0.97,0.206,0.77,1.67,0.63,1.09,0.60,1.43,1.88,0.20
2,Red Lentil (Extruded),95.41,1.08,26.86,3.30,0.96,1.47,4.38,0.79,0.97,...,1.18,0.220,0.96,1.88,0.68,1.33,0.79,1.81,2.01,0.22
3,Red Lentil (Cooked),99.57,1.62,26.62,3.31,0.96,1.56,4.41,0.91,0.85,...,1.21,0.210,1.03,2.19,0.71,1.43,0.77,1.83,2.28,0.22
4,Red Lentil (Baked),97.49,2.34,25.93,3.25,1.00,1.54,4.47,0.96,1.00,...,1.13,0.190,1.00,2.05,0.67,1.26,0.80,1.57,2.30,0.20
5,Green Lentil (Untreated),91.38,1.13,23.93,2.81,0.83,1.31,3.86,0.90,0.87,...,1.02,0.190,0.89,1.75,0.67,1.17,0.66,1.61,2.22,0.18
6,Green Lentil (Extruded),95.13,1.48,24.65,3.05,0.89,1.36,4.07,0.73,0.91,...,1.10,0.200,1.02,1.93,0.64,1.24,0.73,2.14,2.11,0.20
7,Green Lentil (Cooked),99.47,2.06,25.67,3.07,0.91,1.45,4.09,0.80,0.90,...,1.19,0.210,2.05,2.05,0.67,1.32,0.71,2.28,2.21,0.21
8,Green Lentil (Baked),97.32,2.21,25.44,2.53,0.79,1.24,3.67,0.53,0.88,...,1.13,0.180,1.69,1.69,0.57,1.06,0.67,1.37,1.88,0.20


In [12]:
# Combining MET and CYS into a new column "MET + CYS", and PHE and TYR into "PHE + TYR"
df1["MET + CYS"] = df1["MET"] + df1["CYS"]
df1["PHE + TYR"] = df1["PHE"] + df1["TYR"]

# Dropping the individual MET, CYS, PHE, and TYR columns for simplicity
df1_final = df1.drop(columns=["MET", "CYS", "PHE", "TYR"])

# Display the updated table
df1_final.head(), df1_final.columns


(                   Sample    %DM   %CF    %CP   ASP   THR   SER    GLU   PRO  \
 0                  Casein  93.56  0.20  86.48  7.78  3.35  5.64  20.05  9.77   
 1  Red Lentil (Untreated)  92.12  1.78  25.13  2.69  0.78  1.21   3.55  0.60   
 2   Red Lentil (Extruded)  95.41  1.08  26.86  3.30  0.96  1.47   4.38  0.79   
 3     Red Lentil (Cooked)  99.57  1.62  26.62  3.31  0.96  1.56   4.41  0.91   
 4      Red Lentil (Baked)  97.49  2.34  25.93  3.25  1.00  1.54   4.47  0.96   
 
     GLY   ALA   VAL   ILE   LEU   HIS   LYS   ARG   TRP  MET + CYS  PHE + TYR  
 0  1.35  3.16  5.02  3.84  8.39  2.74  6.96  3.12  1.08      2.230       9.42  
 1  0.87  0.98  0.97  0.77  1.67  0.60  1.43  1.88  0.20      0.426       1.72  
 2  0.97  1.25  1.18  0.96  1.88  0.79  1.81  2.01  0.22      0.460       2.01  
 3  0.85  1.23  1.21  1.03  2.19  0.77  1.83  2.28  0.22      0.450       2.14  
 4  1.00  1.29  1.13  1.00  2.05  0.80  1.57  2.30  0.20      0.390       1.93  ,
 Index(['Sample', '%DM', 

In [13]:
df1_final

,Sample,%DM,%CF,%CP,ASP,THR,SER,GLU,PRO,GLY,ALA,VAL,ILE,LEU,HIS,LYS,ARG,TRP,MET + CYS,PHE + TYR
0,Casein,93.56,0.20,86.48,7.78,3.35,5.64,20.05,9.77,1.35,3.16,5.02,3.84,8.39,2.74,6.96,3.12,1.08,2.230,9.42
1,Red Lentil (Untreated),92.12,1.78,25.13,2.69,0.78,1.21,3.55,0.60,0.87,0.98,0.97,0.77,1.67,0.60,1.43,1.88,0.20,0.426,1.72
2,Red Lentil (Extruded),95.41,1.08,26.86,3.30,0.96,1.47,4.38,0.79,0.97,1.25,1.18,0.96,1.88,0.79,1.81,2.01,0.22,0.460,2.01
3,Red Lentil (Cooked),99.57,1.62,26.62,3.31,0.96,1.56,4.41,0.91,0.85,1.23,1.21,1.03,2.19,0.77,1.83,2.28,0.22,0.450,2.14
4,Red Lentil (Baked),97.49,2.34,25.93,3.25,1.00,1.54,4.47,0.96,1.00,1.29,1.13,1.00,2.05,0.80,1.57,2.30,0.20,0.390,1.93
5,Green Lentil (Untreated),91.38,1.13,23.93,2.81,0.83,1.31,3.86,0.90,0.87,1.15,1.02,0.89,1.75,0.66,1.61,2.22,0.18,0.390,1.84
6,Green Lentil (Extruded),95.13,1.48,24.65,3.05,0.89,1.36,4.07,0.73,0.91,1.20,1.10,1.02,1.93,0.73,2.14,2.11,0.20,0.400,1.88
7,Green Lentil (Cooked),99.47,2.06,25.67,3.07,0.91,1.45,4.09,0.80,0.90,1.20,1.19,2.05,2.05,0.71,2.28,2.21,0.21,0.410,1.99
8,Green Lentil (Baked),97.32,2.21,25.44,2.53,0.79,1.24,3.67,0.53,0.88,1.14,1.13,1.69,1.69,0.67,1.37,1.88,0.20,0.360,1.63


In [14]:
!pip install pytesseract
!pip install pillow

  Using cached pytesseract-0.3.13-py3-none-any.whl.metadata (11 kB)
Using cached pytesseract-0.3.13-py3-none-any.whl (14 kB)



[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import pytesseract
import os
# Configure pytesseract to point to the location of the Tesseract executable
# Replace with the correct path for your OS
pytesseract.pytesseract.tesseract_cmd = r'/usr/bin/tesseract'  # or the relevant path on your system
# Check if the path was set correctly.
print(pytesseract.pytesseract.tesseract_cmd)
print(os.environ.get("TESSDATA_PREFIX"))

/usr/bin/tesseract
None


In [16]:
import pandas as pd

# Data for the first table
data2 = {
    "Sample": [
        "Casein",
        "Red Lentil (Extruded)",
        "Red Lentil (Cooked)",
        "Red Lentil (Baked)",
        "Green Lentil (Extruded)",
        "Green Lentil (Cooked)",
        "Green Lentil (Baked)"
    ],
    "Adj. PER": [2.50, 1.05, 1.14, 0.79, 1.08, 0.98, 0.88],
    "AAS": [1.03, 0.68, 0.63, 0.61, 0.66, 0.61, 0.57],
    "%TPD": [96.11, 92.38, 90.95, 88.80, 86.02, 86.42, 83.05],
    "IVPD": [91.36, 88.01, 84.67, 85.03, 84.30, 84.03, 79.33],
    "PDCAAS": [99.09, 63.01, 57.40, 53.84, 57.09, 52.92, 47.14],
    "In Vitro PDCAAS": [94.19, 60.03, 53.43, 51.55, 55.95, 51.46, 45.03]
}

# Creating DataFrame for table 1
df2 = pd.DataFrame(data2)
print("Table 1:")
print(df2)


Table 1:
                    Sample  Adj. PER   AAS   %TPD   IVPD  PDCAAS  \
0                   Casein      2.50  1.03  96.11  91.36   99.09   
1    Red Lentil (Extruded)      1.05  0.68  92.38  88.01   63.01   
2      Red Lentil (Cooked)      1.14  0.63  90.95  84.67   57.40   
3       Red Lentil (Baked)      0.79  0.61  88.80  85.03   53.84   
4  Green Lentil (Extruded)      1.08  0.66  86.02  84.30   57.09   
5    Green Lentil (Cooked)      0.98  0.61  86.42  84.03   52.92   
6     Green Lentil (Baked)      0.88  0.57  83.05  79.33   47.14   

   In Vitro PDCAAS  
0            94.19  
1            60.03  
2            53.43  
3            51.55  
4            55.95  
5            51.46  
6            45.03  


In [17]:
df2

,Sample,Adj. PER,AAS,%TPD,IVPD,PDCAAS,In Vitro PDCAAS
0,Casein,2.50,1.03,96.11,91.36,99.09,94.19
1,Red Lentil (Extruded),1.05,0.68,92.38,88.01,63.01,60.03
2,Red Lentil (Cooked),1.14,0.63,90.95,84.67,57.40,53.43
3,Red Lentil (Baked),0.79,0.61,88.80,85.03,53.84,51.55
4,Green Lentil (Extruded),1.08,0.66,86.02,84.30,57.09,55.95
5,Green Lentil (Cooked),0.98,0.61,86.42,84.03,52.92,51.46
6,Green Lentil (Baked),0.88,0.57,83.05,79.33,47.14,45.03


In [18]:
columns_to_drop=['PDCAAS','In Vitro PDCAAS']

df2_final=df2.drop(columns_to_drop,axis=1)

In [19]:
df2_final

,Sample,Adj. PER,AAS,%TPD,IVPD
0,Casein,2.50,1.03,96.11,91.36
1,Red Lentil (Extruded),1.05,0.68,92.38,88.01
2,Red Lentil (Cooked),1.14,0.63,90.95,84.67
3,Red Lentil (Baked),0.79,0.61,88.80,85.03
4,Green Lentil (Extruded),1.08,0.66,86.02,84.30
5,Green Lentil (Cooked),0.98,0.61,86.42,84.03
6,Green Lentil (Baked),0.88,0.57,83.05,79.33


In [20]:
# Select the essential amino acids columns
essential_amino_acids = ['THR', 'SER', 'GLU', 'PRO', 'GLY', 'ALA',
                         'VAL', 'ILE', 'LEU', 'HIS', 'LYS', 'ARG', 'TRP',
                         'MET + CYS', 'PHE + TYR'] # Assuming these are your essential amino acids

# Create a new DataFrame to store the results
df_essential_amino_acids = df1[['Sample']].copy()  # Start with 'Sample' column

# Calculate mg/g protein for each essential amino acid
for amino_acid in essential_amino_acids:
    df_essential_amino_acids[amino_acid] = (df1[amino_acid] * 1000) / df1['%CP']

# Display the new DataFrame
df_essential_amino_acids

,Sample,THR,SER,GLU,PRO,GLY,ALA,VAL,ILE,LEU,HIS,LYS,ARG,TRP,MET + CYS,PHE + TYR
0,Casein,38.737280,65.217391,231.845513,112.974098,15.610546,36.540241,58.048104,44.403330,97.016651,31.683626,80.481036,36.077706,12.488437,25.786309,108.926920
1,Red Lentil (Untreated),31.038599,48.149622,141.265420,23.875846,34.619976,38.997214,38.599284,30.640669,66.454437,23.875846,56.904099,74.810983,7.958615,16.951850,68.444091
2,Red Lentil (Extruded),35.740879,54.728220,163.067759,29.411765,36.113179,46.537602,43.931497,35.740879,69.992554,29.411765,67.386448,74.832465,8.190618,17.125838,74.832465
3,Red Lentil (Cooked),36.063110,58.602554,165.664914,34.184823,31.930879,46.205860,45.454545,38.692712,82.268971,28.925620,68.745304,85.649887,8.264463,16.904583,80.390684
4,Red Lentil (Baked),38.565368,59.390667,172.387196,37.022754,38.565368,49.749325,43.578866,38.565368,79.059005,30.852295,60.547628,88.700347,7.713074,15.040494,74.431161
5,Green Lentil (Untreated),34.684496,54.743000,161.303803,37.609695,36.356038,48.056832,42.624321,37.191809,73.129962,27.580443,67.279565,92.770581,7.521939,16.297534,76.890932
6,Green Lentil (Extruded),36.105477,55.172414,165.111562,29.614604,36.916836,48.681542,44.624746,41.379310,78.296146,29.614604,86.815416,85.598377,8.113590,16.227181,76.267748
7,Green Lentil (Cooked),35.449942,56.486171,159.329957,31.164784,35.060382,46.747176,46.357616,79.859758,79.859758,27.658746,88.819634,86.092715,8.180756,15.971952,77.522400
8,Green Lentil (Baked),31.053459,48.742138,144.261006,20.833333,34.591195,44.811321,44.418239,66.430818,66.430818,26.336478,53.852201,73.899371,7.861635,14.150943,64.072327


In [21]:
# df_amino_acids = pd.DataFrame(amino_acids_data)
# df_amino_acids['Sample'] = df['Sample'] 
# for amino_acid in df_amino_acids.columns[:-1]:  # Exclude the 'Sample' column
#     df_amino_acids[f'{amino_acid} (mg/g protein)'] = (df_amino_acids[amino_acid] * 1000) / protein_percentages

# # Display the results
# print(df_amino_acids)

In [22]:
df_essential_amino_acids.columns

Index(['Sample', 'THR', 'SER', 'GLU', 'PRO', 'GLY', 'ALA', 'VAL', 'ILE', 'LEU',
       'HIS', 'LYS', 'ARG', 'TRP', 'MET + CYS', 'PHE + TYR'],
      dtype='object')

In [23]:
columns_to_remove=['SER', 'GLU', 'PRO', 'GLY', 'ALA','ARG']

In [24]:
df_essential_amino_acids=df_essential_amino_acids.drop(columns_to_remove,axis=1)

In [25]:
df_essential_amino_acids

,Sample,THR,VAL,ILE,LEU,HIS,LYS,TRP,MET + CYS,PHE + TYR
0,Casein,38.737280,58.048104,44.403330,97.016651,31.683626,80.481036,12.488437,25.786309,108.926920
1,Red Lentil (Untreated),31.038599,38.599284,30.640669,66.454437,23.875846,56.904099,7.958615,16.951850,68.444091
2,Red Lentil (Extruded),35.740879,43.931497,35.740879,69.992554,29.411765,67.386448,8.190618,17.125838,74.832465
3,Red Lentil (Cooked),36.063110,45.454545,38.692712,82.268971,28.925620,68.745304,8.264463,16.904583,80.390684
4,Red Lentil (Baked),38.565368,43.578866,38.565368,79.059005,30.852295,60.547628,7.713074,15.040494,74.431161
5,Green Lentil (Untreated),34.684496,42.624321,37.191809,73.129962,27.580443,67.279565,7.521939,16.297534,76.890932
6,Green Lentil (Extruded),36.105477,44.624746,41.379310,78.296146,29.614604,86.815416,8.113590,16.227181,76.267748
7,Green Lentil (Cooked),35.449942,46.357616,79.859758,79.859758,27.658746,88.819634,8.180756,15.971952,77.522400
8,Green Lentil (Baked),31.053459,44.418239,66.430818,66.430818,26.336478,53.852201,7.861635,14.150943,64.072327


In [26]:
import pandas as pd

# Data for the table
data = {
    "THR": [28],
    "VAL": [25],
    "MET + CYS": [22],
    "ILE": [28],
    "LEU": [44],
    "PHE + TYR": [22],
    "HIS": [19],
    "LYS": [44],
    "TRP": [9]
}

# Creating the DataFrame
df = pd.DataFrame(data)

# Displaying the DataFrame
print(df)


   THR  VAL  MET + CYS  ILE  LEU  PHE + TYR  HIS  LYS  TRP
0   28   25         22   28   44         22   19   44    9


In [27]:
df

,THR,VAL,MET + CYS,ILE,LEU,PHE + TYR,HIS,LYS,TRP
0,28,25,22,28,44,22,19,44,9


In [28]:
essential_amino_acids = ['THR','VAL', 'ILE', 'LEU', 'HIS', 'LYS', 'TRP',
                         'MET + CYS', 'PHE + TYR']
for amino_acid in essential_amino_acids:
    ref=df[amino_acid].iloc[0]
    df_essential_amino_acids[amino_acid] = (df_essential_amino_acids[amino_acid] / ref)

In [29]:
ref

22

In [30]:
df_essential_amino_acids

,Sample,THR,VAL,ILE,LEU,HIS,LYS,TRP,MET + CYS,PHE + TYR
0,Casein,1.383474,2.321924,1.585833,2.204924,1.667559,1.829114,1.387604,1.172105,4.951224
1,Red Lentil (Untreated),1.108521,1.543971,1.094310,1.510328,1.256623,1.293275,0.884291,0.770539,3.111095
2,Red Lentil (Extruded),1.276460,1.757260,1.276460,1.590740,1.547988,1.531510,0.910069,0.778447,3.401476
3,Red Lentil (Cooked),1.287968,1.818182,1.381883,1.869749,1.522401,1.562393,0.918274,0.768390,3.654122
4,Red Lentil (Baked),1.377335,1.743155,1.377335,1.796796,1.623805,1.376082,0.857008,0.683659,3.383235
5,Green Lentil (Untreated),1.238732,1.704973,1.328279,1.662045,1.451602,1.529081,0.835771,0.740797,3.495042
6,Green Lentil (Extruded),1.289481,1.784990,1.477833,1.779458,1.558663,1.973078,0.901510,0.737599,3.466716
7,Green Lentil (Cooked),1.266069,1.854305,2.852134,1.814995,1.455723,2.018628,0.908973,0.725998,3.523745
8,Green Lentil (Baked),1.109052,1.776730,2.372529,1.509791,1.386130,1.223914,0.873515,0.643225,2.912379


In [31]:
# Convert all columns except 'Sample' to numeric, handling errors
for col in df_essential_amino_acids.columns:
    if col != 'Sample':
        df_essential_amino_acids[col] = pd.to_numeric(df_essential_amino_acids[col], errors='coerce')

# Calculate the 'Limiting Amino Acid' and its corresponding column name
df_essential_amino_acids['AAS'] = df_essential_amino_acids.min(axis=1, numeric_only=True)
df_essential_amino_acids['Limiting Amino Acid'] = df_essential_amino_acids.drop(columns=['Sample']).idxmin(axis=1)

In [32]:
df_essential_amino_acids

,Sample,THR,VAL,ILE,LEU,HIS,LYS,TRP,MET + CYS,PHE + TYR,AAS,Limiting Amino Acid
0,Casein,1.383474,2.321924,1.585833,2.204924,1.667559,1.829114,1.387604,1.172105,4.951224,1.172105,MET + CYS
1,Red Lentil (Untreated),1.108521,1.543971,1.094310,1.510328,1.256623,1.293275,0.884291,0.770539,3.111095,0.770539,MET + CYS
2,Red Lentil (Extruded),1.276460,1.757260,1.276460,1.590740,1.547988,1.531510,0.910069,0.778447,3.401476,0.778447,MET + CYS
3,Red Lentil (Cooked),1.287968,1.818182,1.381883,1.869749,1.522401,1.562393,0.918274,0.768390,3.654122,0.768390,MET + CYS
4,Red Lentil (Baked),1.377335,1.743155,1.377335,1.796796,1.623805,1.376082,0.857008,0.683659,3.383235,0.683659,MET + CYS
5,Green Lentil (Untreated),1.238732,1.704973,1.328279,1.662045,1.451602,1.529081,0.835771,0.740797,3.495042,0.740797,MET + CYS
6,Green Lentil (Extruded),1.289481,1.784990,1.477833,1.779458,1.558663,1.973078,0.901510,0.737599,3.466716,0.737599,MET + CYS
7,Green Lentil (Cooked),1.266069,1.854305,2.852134,1.814995,1.455723,2.018628,0.908973,0.725998,3.523745,0.725998,MET + CYS
8,Green Lentil (Baked),1.109052,1.776730,2.372529,1.509791,1.386130,1.223914,0.873515,0.643225,2.912379,0.643225,MET + CYS


In [33]:
new_df=df1[['Sample']].copy()

In [34]:
new_df=pd.merge(new_df,df2_final[['Sample','%TPD','IVPD']],on='Sample', how='left')

In [35]:
new_df=pd.merge(new_df,df_essential_amino_acids[['Sample','AAS']],on='Sample',how='left')

In [36]:
new_df

,Sample,%TPD,IVPD,AAS
0,Casein,96.11,91.36,1.172105
1,Red Lentil (Untreated),NaN,NaN,0.770539
2,Red Lentil (Extruded),92.38,88.01,0.778447
3,Red Lentil (Cooked),90.95,84.67,0.768390
4,Red Lentil (Baked),88.80,85.03,0.683659
5,Green Lentil (Untreated),NaN,NaN,0.740797
6,Green Lentil (Extruded),86.02,84.30,0.737599
7,Green Lentil (Cooked),86.42,84.03,0.725998
8,Green Lentil (Baked),83.05,79.33,0.643225


In [37]:
new_df['PDCAAS']=new_df['AAS']*new_df['%TPD']

In [38]:
new_df['IVPDCAAS']=new_df['AAS']*new_df['IVPD']

In [39]:
new_df

,Sample,%TPD,IVPD,AAS,PDCAAS,IVPDCAAS
0,Casein,96.11,91.36,1.172105,112.651007,107.083509
1,Red Lentil (Untreated),NaN,NaN,0.770539,NaN,NaN
2,Red Lentil (Extruded),92.38,88.01,0.778447,71.912949,68.511135
3,Red Lentil (Cooked),90.95,84.67,0.768390,69.885083,65.059593
4,Red Lentil (Baked),88.80,85.03,0.683659,60.708902,58.131508
5,Green Lentil (Untreated),NaN,NaN,0.740797,NaN,NaN
6,Green Lentil (Extruded),86.02,84.30,0.737599,63.448276,62.179605
7,Green Lentil (Cooked),86.42,84.03,0.725998,62.740730,61.005595
8,Green Lentil (Baked),83.05,79.33,0.643225,53.419811,51.027015


In [40]:
df1_final

,Sample,%DM,%CF,%CP,ASP,THR,SER,GLU,PRO,GLY,ALA,VAL,ILE,LEU,HIS,LYS,ARG,TRP,MET + CYS,PHE + TYR
0,Casein,93.56,0.20,86.48,7.78,3.35,5.64,20.05,9.77,1.35,3.16,5.02,3.84,8.39,2.74,6.96,3.12,1.08,2.230,9.42
1,Red Lentil (Untreated),92.12,1.78,25.13,2.69,0.78,1.21,3.55,0.60,0.87,0.98,0.97,0.77,1.67,0.60,1.43,1.88,0.20,0.426,1.72
2,Red Lentil (Extruded),95.41,1.08,26.86,3.30,0.96,1.47,4.38,0.79,0.97,1.25,1.18,0.96,1.88,0.79,1.81,2.01,0.22,0.460,2.01
3,Red Lentil (Cooked),99.57,1.62,26.62,3.31,0.96,1.56,4.41,0.91,0.85,1.23,1.21,1.03,2.19,0.77,1.83,2.28,0.22,0.450,2.14
4,Red Lentil (Baked),97.49,2.34,25.93,3.25,1.00,1.54,4.47,0.96,1.00,1.29,1.13,1.00,2.05,0.80,1.57,2.30,0.20,0.390,1.93
5,Green Lentil (Untreated),91.38,1.13,23.93,2.81,0.83,1.31,3.86,0.90,0.87,1.15,1.02,0.89,1.75,0.66,1.61,2.22,0.18,0.390,1.84
6,Green Lentil (Extruded),95.13,1.48,24.65,3.05,0.89,1.36,4.07,0.73,0.91,1.20,1.10,1.02,1.93,0.73,2.14,2.11,0.20,0.400,1.88
7,Green Lentil (Cooked),99.47,2.06,25.67,3.07,0.91,1.45,4.09,0.80,0.90,1.20,1.19,2.05,2.05,0.71,2.28,2.21,0.21,0.410,1.99
8,Green Lentil (Baked),97.32,2.21,25.44,2.53,0.79,1.24,3.67,0.53,0.88,1.14,1.13,1.69,1.69,0.67,1.37,1.88,0.20,0.360,1.63


In [41]:
new_df=pd.merge(new_df,df1_final[['Sample','%CP']],on='Sample', how='left')

In [42]:
new_df

,Sample,%TPD,IVPD,AAS,PDCAAS,IVPDCAAS,%CP
0,Casein,96.11,91.36,1.172105,112.651007,107.083509,86.48
1,Red Lentil (Untreated),NaN,NaN,0.770539,NaN,NaN,25.13
2,Red Lentil (Extruded),92.38,88.01,0.778447,71.912949,68.511135,26.86
3,Red Lentil (Cooked),90.95,84.67,0.768390,69.885083,65.059593,26.62
4,Red Lentil (Baked),88.80,85.03,0.683659,60.708902,58.131508,25.93
5,Green Lentil (Untreated),NaN,NaN,0.740797,NaN,NaN,23.93
6,Green Lentil (Extruded),86.02,84.30,0.737599,63.448276,62.179605,24.65
7,Green Lentil (Cooked),86.42,84.03,0.725998,62.740730,61.005595,25.67
8,Green Lentil (Baked),83.05,79.33,0.643225,53.419811,51.027015,25.44


In [2]:
Title = "In vitro versus in vivo protein digestibility techniques for calculating PDCAAS (protein digestibility-corrected amino acid score) applied to chickpea fractions"
DOI =  "10.1016/j.foodres.2016.10.005"

In [57]:


# from bs4 import BeautifulSoup
# with open("file.html", "r") as f:
#     html = f.read()
# soup = BeautifulSoup(html, 'html.parser')

# result_dict = {}

# for table_id in map(chr, range(ord('a'), ord('x') + 1)):
#     tables = soup.find_all('table', {'id': table_id})  
#     if tables:
#         print(f"Found {len(tables)} table(s) with ID: {table_id}")  
#     for table in tables:
#         rows = table.find_all('tr')
#         for row in rows:
#             print(f"Row text: {row.get_text().strip()}") 
#             if "peas and lentils" in row.get_text().lower():  
#                 print("Match found for 'peas and lentils'")  
#                 tds = row.find_all('td')
#                 if len(tds) >= 3: 
#                     second_td_text = tds[1].get_text().strip() 
#                     third_td_text = tds[2].get_text().strip()  
#                     print(f"Adding to dictionary: {second_td_text} -> {third_td_text}")  
#                     result_dict[second_td_text] = third_td_text

# # Print the result dictionary
# print("Result dictionary:", result_dict)
encodings = ["utf-8", "ISO-8859-1", "Windows-1252"]
from bs4 import BeautifulSoup
import json

encodings = ["ISO-8859-1", "Windows-1252"]

for encoding in encodings:
    try:
        with open("file.html", "r", encoding=encoding) as f:
            html = f.read().strip()
        # print(f"Successfully read file using {encoding}")
        break
    except UnicodeDecodeError:
        print(f" Failed with {encoding}, trying next...")
else:
    print("Unable to read the file with known encodings!")
    exit()

soup = BeautifulSoup(html, 'html.parser')
result_dict = {}

for table in soup.find_all('table'):
    table_id = table.get('id', 'No ID')
    print(f"📌 Processing Table ID: {table_id}")

    for row in table.find_all('tr'):
        tds = row.find_all('td')
        if len(tds) >= 3:  
            second_col = tds[1].get_text(strip=True)  # Key
            third_col = tds[2].get_text(strip=True)   # Value

            result_dict[second_col] = third_col
            print(f"{second_col}: {third_col}")
output_path = "output.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(result_dict, f, indent=4, ensure_ascii=False)

print(f"Final Result Dictionary saved to {output_path}")


📌 Processing Table ID: a
Bread, including garlic bread and other bread with add-ins, such as raisins, olives and cheese, but excluding sweet quick-type bread: 75 g
MM: amount in grams needed to prepare RAHM: fraction of a cup or of the package to prepare the RA (according to directions for use): fraction cup (# g)fraction package (# g)
Tea biscuits, scones, rolls, buns, Yorkshire pudding, English muffins, croissants, tortillas, pita, soft bread sticks, soft pretzels and corn bread, with or without filling or add-ins, such as raisins, olives and cheese: 55 g
MM: amount in grams needed to prepare RAHM: the number of tablespoons or fraction of a cup or of the package to prepare the RA (according to directions for use): # tbsp (# g)fraction cup (# g)fraction package (# g)
Bagels, naan, flat bread: 85 g
Brownies, dessert squares and bars: 40 g
MM: amount in grams needed to prepare RAHM: the number of tablespoons or fraction of a cup or of the package to prepare the RA (according to directio

In [58]:

first_list = list(data1.values())[0]
print(first_list)

['Casein', 'Red Lentil (Untreated)', 'Red Lentil (Extruded)', 'Red Lentil (Cooked)', 'Red Lentil (Baked)', 'Green Lentil (Untreated)', 'Green Lentil (Extruded)', 'Green Lentil (Cooked)', 'Green Lentil (Baked)']


In [93]:
Title_ = "margarine"

In [94]:
title_lower = Title_.lower()
# Find matching values in dictionary
matched_values = [value for key, value in result_dict.items() if title_lower in key.lower()]

# Print matched values
if matched_values:
    print(" Matched Values:")
    for value in matched_values:
        print(value)
else:
    print("No matches found!")


 Matched Values:
10 g


In [54]:
rac = int(racc)

In [55]:
new_df['CCS']=(new_df['%CP']/100)*(new_df['PDCAAS']/100)*rac

In [56]:
new_df

,Sample,%TPD,IVPD,AAS,PDCAAS,IVPDCAAS,%CP,CCS
0,Casein,96.11,91.36,1.172105,112.651007,107.083509,86.48,34.097207
1,Red Lentil (Untreated),NaN,NaN,0.770539,NaN,NaN,25.13,NaN
2,Red Lentil (Extruded),92.38,88.01,0.778447,71.912949,68.511135,26.86,6.760536
3,Red Lentil (Cooked),90.95,84.67,0.768390,69.885083,65.059593,26.62,6.511193
4,Red Lentil (Baked),88.80,85.03,0.683659,60.708902,58.131508,25.93,5.509636
5,Green Lentil (Untreated),NaN,NaN,0.740797,NaN,NaN,23.93,NaN
6,Green Lentil (Extruded),86.02,84.30,0.737599,63.448276,62.179605,24.65,5.474000
7,Green Lentil (Cooked),86.42,84.03,0.725998,62.740730,61.005595,25.67,5.636941
8,Green Lentil (Baked),83.05,79.33,0.643225,53.419811,51.027015,25.44,4.756500


In [48]:
import numpy as np

In [57]:
new_df['Content Claim Status']=np.where(new_df['CCS']>=10, 'Excellent Source of protein',np.where(new_df['CCS']<5,'Poor Source of Protein','Good Source of Protein'))

In [58]:
new_df

,Sample,%TPD,IVPD,AAS,PDCAAS,IVPDCAAS,%CP,CCS,Content Claim Status
0,Casein,96.11,91.36,1.172105,112.651007,107.083509,86.48,34.097207,Excellent Source of protein
1,Red Lentil (Untreated),NaN,NaN,0.770539,NaN,NaN,25.13,NaN,Good Source of Protein
2,Red Lentil (Extruded),92.38,88.01,0.778447,71.912949,68.511135,26.86,6.760536,Good Source of Protein
3,Red Lentil (Cooked),90.95,84.67,0.768390,69.885083,65.059593,26.62,6.511193,Good Source of Protein
4,Red Lentil (Baked),88.80,85.03,0.683659,60.708902,58.131508,25.93,5.509636,Good Source of Protein
5,Green Lentil (Untreated),NaN,NaN,0.740797,NaN,NaN,23.93,NaN,Good Source of Protein
6,Green Lentil (Extruded),86.02,84.30,0.737599,63.448276,62.179605,24.65,5.474000,Good Source of Protein
7,Green Lentil (Cooked),86.42,84.03,0.725998,62.740730,61.005595,25.67,5.636941,Good Source of Protein
8,Green Lentil (Baked),83.05,79.33,0.643225,53.419811,51.027015,25.44,4.756500,Poor Source of Protein
